Desigualdades en el acceso a hospitalización compleja:

● ¿Existen diferencias regionales en el acceso a hospitalizaciones de alta complejidad en
el sistema público de salud?

● ¿La distribución de pacientes de alta complejidad se concentra en ciertos hospitales?

Vamos a filtrar todos los hospitales con mortalidad hospitalaria alta. 
Luego chequear Hospitales con alta frecuencia de traslado y revisar donde están siendo trasladados. 
Para así entender que hospital está generando mas traslados y entender que esta sucediendo en ese hospital en particular.


# ¿Cuál es la relación entre los altos índices de mortalidad hospitalaria y los patrones de derivación de pacientes en la red pública de salud, y qué características clínicas o de gestión explican el comportamiento de los hospitales con mayor emisión de traslados?

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import glob as gb
import os
import csv

In [ ]:
# Definimos las columnas que realmente nos interesan para evitar cargar datos innecesarios
columnas_necesarias = [
    'COD_HOSPITAL', 'SEXO', 'FECHA_INGRESO', 'DIAGNOSTICO1', 
    'PROCEDIMIENTO1', 'IR_29301_COD_GRD', 'IR_29301_PESO'
]

# 1. Buscamos todos los archivos .csv dentro de la carpeta llamada 'csv'
ruta_archivos = 'csv/*.csv'
lista_archivos = gb.glob(ruta_archivos)

# Lista vacía donde guardaremos cada DataFrame temporalmente
dataframes = []

print(f"Se encontraron {len(lista_archivos)} archivos. Comenzando la lectura...\n")

# 2. Iteramos sobre cada archivo encontrado
for archivo in lista_archivos:
    print(f"Procesando: {archivo}...")
    
    # Usamos la configuración robusta que funcionó
    try:
        df_temp = pd.read_csv(
            archivo, 
            sep='|', 
            decimal=',',
            encoding='utf-16',          
            quoting=csv.QUOTE_NONE,
            usecols=columnas_necesarias,  
            index_col=False,            
            on_bad_lines='warn',        
            low_memory=False
        )
    except UnicodeError:
        # Plan B por si algún archivo en la carpeta tiene codificación distinta
        df_temp = pd.read_csv(
            archivo, 
            sep='|', 
            decimal=',',
            encoding='utf-8-sig',       
            quoting=csv.QUOTE_NONE,
            usecols=columnas_necesarias, 
            index_col=False,
            on_bad_lines='warn',
            low_memory=False
        )
    
    # Agregamos el DataFrame a nuestra lista
    dataframes.append(df_temp)

# 3. Concatenamos todo de una sola vez
# ignore_index=True es clave: resetea el conteo de filas (0, 1, 2...) para que no se repitan
df_consolidado = pd.concat(dataframes, ignore_index=True)

print("\n¡Proceso terminado!")
print(f"El DataFrame final consolidado tiene {df_consolidado.shape[0]} filas y {df_consolidado.shape[1]} columnas.")

Se encontraron 6 archivos. Comenzando la lectura...

Procesando: csv\GRD_PUBLICO_2019.csv...


ParserError: Error tokenizing data. C error: out of memory

In [ ]:
# Intentaremos leer con la configuración más robusta posible
try:
    df = pd.read_csv(
        'csv\GRD_PUBLICO_2022.csv', 
        sep='|', 
        decimal=',',
        encoding='utf-16',          # MAGIA 1: Lee formatos pesados de Windows/SQL
        nrows=100,
        quoting=csv.QUOTE_NONE,     # MAGIA 2: Ignora comillas mal cerradas
        index_col=False,            # Evita que columnas vacías al final desfasen los datos
        on_bad_lines='warn',        # Si hay una línea rota, te avisa en vez de colapsar
        low_memory=False
    )
    print("¡Lectura exitosa con UTF-16!")
except UnicodeError:
    # Si falla el UTF-16, intentamos con el otro formato clásico con BOM
    df = pd.read_csv(
        'csv\GRD_PUBLICO_2022.csv', 
        sep='|', 
        decimal=',',
        encoding='utf-8-sig',       # Variante de UTF-8 que remueve bytes invisibles al inicio
        nrows=100,
        quoting=csv.QUOTE_NONE, 
        index_col=False,
        on_bad_lines='warn',
        low_memory=False
    )
    print("¡Lectura exitosa con UTF-8-SIG!")

# Veamos qué estructura reconoció realmente pandas
print(f"Filas y columnas encontradas: {df.shape}")

¡Lectura exitosa con UTF-16!
Filas y columnas encontradas: (100, 129)


In [86]:
df.columns.to_list()

['COD_HOSPITAL',
 'CIP_ENCRIPTADO',
 'SEXO',
 'FECHA_NACIMIENTO',
 'ETNIA',
 'PROVINCIA',
 'COMUNA',
 'NACIONALIDAD',
 'PREVISION',
 'SERVICIO_SALUD',
 'TIPO_PROCEDENCIA',
 'TIPO_INGRESO',
 'ESPECIALIDAD_MEDICA',
 'TIPO_ACTIVIDAD',
 'FECHA_INGRESO',
 'SERVICIOINGRESO',
 'FECHATRASLADO1',
 'SERVICIOTRASLADO1',
 'FECHATRASLADO2',
 'SERVICIOTRASLADO2',
 'FECHATRASLADO3',
 'SERVICIOTRASLADO3',
 'FECHATRASLADO4',
 'SERVICIOTRASLADO4',
 'FECHATRASLADO5',
 'SERVICIOTRASLADO5',
 'FECHATRASLADO6',
 'SERVICIOTRASLADO6',
 'FECHATRASLADO7',
 'SERVICIOTRASLADO7',
 'FECHATRASLADO8',
 'SERVICIOTRASLADO8',
 'FECHATRASLADO9',
 'SERVICIOTRASLADO9',
 'FECHAALTA',
 'SERVICIOALTA',
 'TIPOALTA',
 'CONDICIONDEALTANEONATO1',
 'PESORN1',
 'SEXORN1',
 'RN1ESTADO',
 'CONDICIONDEALTANEONATO2',
 'PESORN2',
 'SEXORN2',
 'RN2ESTADO',
 'CONDICIONDEALTANEONATO3',
 'PESORN3',
 'SEXORN3',
 'RN3ESTADO',
 'CONDICIONDEALTANEONATO4',
 'PESORN4',
 'SEXORN4',
 'RN4ESTADO',
 'DIAGNOSTICO1',
 'DIAGNOSTICO2',
 'DIAGNOSTICO3',
 '